## LSTM (Long Short-Term Memory)

A rede do tipo Long Short-term Memory (LSTM)é uma modificação da rede neural recorrente (RNN) padrão destinada a mitigar o problema do desaparecimento do gradiente comumente encontrado pelas RNNs tradicionais. Seu objetivo é fornecer uma memória de curto prazo para RNN que possa durar milhares de passos de tempo (daí o nome "Longa memória de curto prazo").

A rede LSTM evita o problema do gradiente evanescente usando células de memória, portas e funções de ativação projetadas para preservar os gradientes à medida que são retropropagados pela rede. O autoloop na célula de memória, a função de ativação sigmoide usada para controlar as portas e a função de ativação usada para produzir a saída final reduzem a possibilidade do "desaparecimento gradual"do gradiente. Além disso, ao contrário das redes recorrentes, as diferentes portas garantem que entradas mais espaçadas (Long Term Memory) sejam armazenadas no vetor *CellState*, enquanto as memórias de curta duração são armazenadas no vetor *hstate* como nas RNN.
<div>
<img src="Files/ChatGPT Image LSTM célula.png" width="500" height="500">   
</div>  
Fonte: Gerado com chatGPT

Uma última porta final de saída (não mostrada acima) é definida como 
$$\hat{y}_t=\sum_t \left(w_y \cdot h_t +b_y\right).$$

* Retropropagação pelo tempo (BPTT)

 A retropropagação inicia, como sempre, pela derivada da função de perdas da saída da célula em respeito à própria saída. A derivada depende obviamente da função *Loss* escolhida.  Temos então. inicialmente, $\frac{\partial L}{\partial \hat{y}}$.
 A derivada com respeito aos pesos são obtidas através do método das derivadas parciais (veja as equações de propagação direta na figura acima):

  $$\frac{\partial L}{\partial w_y}= \sum_t \frac{\partial L}{\partial \hat{y}_t} \circ \frac{\partial \hat{y}_t}{\partial w_y}=\sum_t\frac{\partial L}{\partial \hat{y}_t} \cdot h_t$$
  $$\frac{\partial L}{\partial b_y}=\sum_t\frac{\partial L}{\partial \hat{y_t}},$$
onde o símbolo $\circ$ é usado para siginificar o produto de Hadamard, ou produto termo a termo.

Agora, para os outros gradientes, precisaremos primeiro determinar os gradientes em relação ao estado oculto $h_t$ e ao estado da célula $C_t$. Entretanto, é importante notar que em uma LSTM , $h_t$ e $C_t$ influenciam o futuro por múltiplos caminhos simultaneamente. Com efeito, $h_t=o_t \circ tanh(C_t)$ e ao mesmo é utilizado em  $h_{t+1}$. Portanto, podemos escrever que

 $$\frac{\partial L}{\partial h_t}=\frac{\partial L}{\partial h_t}+\frac{\partial L_{next}}{\partial h_t}$$
  onde a primeira derivada no lado direito é o gradiente da função *Loss* em relação ao estado presente e a segunda é o gradiente  da função *Loss* no instante $t+1$ em relação à ${h_t}$. Assim,
  $$\frac{\partial L}{\partial h_t}=\frac{\partial L}{\partial \hat{y}_t}\cdot w_y+\frac{\partial L_{next}}{\partial h_t}$$

Seja $z_t$ a concatenação do estado oculto na saída do passo anterior e da entrada no passo atual, como mostrado nas expressões da figura acima, ou seja, $z_t=\begin{bmatrix}h_{t-1} \\ x_t\end{bmatrix}$. $z_t$ está presente nas definições de $f_t,\;i_t,\;\tilde{c}_t\;\text{e}\;o_t$. Daí que 
$$\frac{\partial L}{\partial z_t}=w_f\cdot\frac{\partial L}{\partial f_t}+w_i\cdot\frac{\partial L}{\partial i_t}+w_c\cdot\frac{\partial L}{\partial \tilde{c}_t}+w_o\cdot\frac{\partial L}{\partial o_t}=\begin{bmatrix}\frac{\partial L}{\partial h_{t-1}} \\ \frac{\partial L}{\partial x_t}\end{bmatrix}$$
e finalmente os primeiros termos deste vetor correspondem à $\frac{\partial L_{next}}{\partial h_t}$. 

Para o estado da célula, $C_t$, temos também que $C_{t+1}=f_{t+1}C_t+ i_{t+1}\tilde{C}_{t+1}$, de forma que além do estado oculto o estado da célula no instante $t+1$ também de pende do estado no instante $t$, e daí
$$\frac{\partial L}{\partial C_t}=\frac{\partial L}{\partial h_t}\circ\frac{\partial h_t}{\partial C_t}+\frac{\partial L_{next}}{\partial C_t},$$
onde $\frac{\partial L_{next}}{\partial C_t}=f_{t+1}\frac{\partial L}{\partial C_{t+1}}.$  Os outros gradientes podem ser agora obtidos com facilidade:


$$\frac{\partial L}{\partial w_o}=\sum_t\left(\frac{\partial L}{\partial h_t}\circ\frac{\partial h_t}{\partial o_t}\circ\frac{\partial o_t}{\partial w_o}\right)$$

$$\frac{\partial L}{\partial b_o}=\sum_t\left(\frac{\partial L}{\partial h_t}\circ\frac{\partial h_t}{\partial o_t}\right)$$

$$\frac{\partial L}{\partial w_f}=\sum_t\left(\frac{\partial L}{\partial C_t}\circ\frac{\partial C_t}{\partial f_t}\circ\frac{\partial f_t}{\partial w_f}\right)$$

$$\frac{\partial L}{\partial b_f}=\sum_t\left(\frac{\partial L}{\partial C_t}\circ\frac{\partial C_t}{\partial b_f}\right)$$

$$\frac{\partial L}{\partial w_i}=\sum_t\left(\frac{\partial L}{\partial C_t}\circ\frac{\partial C_t}{\partial i_t}\circ\frac{\partial i_t}{\partial w_i}\right)$$

$$\frac{\partial L}{\partial b_i}=\sum_t\left(\frac{\partial L}{\partial C_t}\circ\frac{\partial C_t}{\partial i_t}\right)$$

$$\frac{\partial L}{\partial w_c}=\sum_t\left(\frac{\partial L}{\partial C_t}\circ\frac{\partial C_t}{\partial \tilde{C}_t}\circ\frac{\partial \tilde{C}_t}{\partial w_c}\right)$$

$$\frac{\partial L}{\partial b_c}=\sum_t\left(\frac{\partial L}{\partial C_t}\circ\frac{\partial C_t}{\partial \tilde{C}_t}\right)$$